#**Evaluation for Bodeoum TV**
This notebook contains all the evaluation processes of mainly sentiment analysis for comments in the Bodeoum TV scandal.

Data set:sentiment_checkpoint_128000.csv this dataset contains 128,000 comments of clean and pre-processed data.

**Shapiro-Wilk Test**
This is to determine if the data is normal or skewed to be able to determine what kind of statistical testing will be utilized for further investigation.

In [9]:
import pandas as pd
from scipy import stats

print("--- SHAPIRO-WILK TEST: SENTIMENT SCORES ---")

sentiment_df = pd.read_csv('sentiment_checkpoint_128000.csv')

sample_size = min(5000, len(sentiment_df))
sentiment_sample = sentiment_df.sample(n=sample_size, random_state=42)

windows = sentiment_sample['window'].dropna().unique()

for window in windows:
    scores = sentiment_sample[sentiment_sample['window'] == window]['sentiment_score']

    # Run the Shapiro-Wilk test
    statistic, p_value = stats.shapiro(scores)

    distribution = "Normally Distributed" if p_value >= 0.05 else "Skewed (Non-Normal)"

    print(f"Window: {window.upper()}")
    print(f"  Test Statistic: {statistic:.4f}")
    print(f"  p-value: {p_value:.4g} -> {distribution}\n")

--- SHAPIRO-WILK TEST: SENTIMENT SCORES ---
Window: POST
  Test Statistic: 0.7256
  p-value: 5.776e-29 -> Skewed (Non-Normal)

Window: CANCEL
  Test Statistic: 0.9144
  p-value: 1.642e-41 -> Skewed (Non-Normal)

Window: PRE
  Test Statistic: 0.9082
  p-value: 4.247e-21 -> Skewed (Non-Normal)



**Kruskal-Wallis H & Dunn's Post-Hoc Test**
Since the data is skewed, a kruskal-wallis H and Dunn's Post-Hoc Test is utilized to determine if sentiments across different windows significantly differ from each other.

In [10]:
!pip install scikit-posthocs

import pandas as pd
from scipy import stats
import scikit_posthocs as sp

print("--- KRUSKAL-WALLIS H TEST ---")

# 1. Load the raw comment dataset
# Make sure the file path matches where you have the CSV saved
sentiment_df = pd.read_csv('sentiment_checkpoint_128000.csv')

# 2. Drop any rows where 'window' or 'sentiment_score' might be missing
clean_df = sentiment_df.dropna(subset=['window', 'sentiment_score'])

# 3. Separate the sentiment scores into three distinct groups based on the window
pre_scores = clean_df[clean_df['window'] == 'pre']['sentiment_score']
cancel_scores = clean_df[clean_df['window'] == 'cancel']['sentiment_score']
post_scores = clean_df[clean_df['window'] == 'post']['sentiment_score']

# 4. Run the Kruskal-Wallis H Test
kw_stat, kw_p_value = stats.kruskal(pre_scores, cancel_scores, post_scores)

print(f"Kruskal-Wallis H-statistic: {kw_stat:.4f}")
print(f"p-value: {kw_p_value:.4g}\n")

# 5. Interpret the Kruskal-Wallis result
if kw_p_value < 0.05:
    print("Result: SIGNIFICANT difference found in sentiment across the windows (p < 0.05).")
    print("Proceeding to Dunn's Post-Hoc Test to see WHICH windows differ...\n")

    print("--- DUNN'S POST-HOC TEST (with Bonferroni correction) ---")

    dunn_results = sp.posthoc_dunn(clean_df, val_col='sentiment_score', group_col='window', p_adjust='bonferroni')

    # Print the resulting p-value matrix
    print(dunn_results)

def calculate_kruskal_eta_sq(data, group_col, value_col):
    groups = [group[value_col].values for name, group in data.groupby(group_col)]
    h_stat, p_val = stats.kruskal(*groups)

    # 2. Get necessary values for the formula
    n = len(data)             # Total number of observations
    k = len(groups)          # Number of groups

    # 3. Calculate Eta-squared
    eta_sq = (h_stat - k + 1) / (n - k)

    return h_stat, p_val, eta_sq

--- KRUSKAL-WALLIS H TEST ---
Kruskal-Wallis H-statistic: 4787.5402
p-value: 0

Result: SIGNIFICANT difference found in sentiment across the windows (p < 0.05).
Proceeding to Dunn's Post-Hoc Test to see WHICH windows differ...

--- DUNN'S POST-HOC TEST (with Bonferroni correction) ---
              cancel  post           pre
cancel  1.000000e+00   0.0  2.625237e-13
post    0.000000e+00   1.0  0.000000e+00
pre     2.625237e-13   0.0  1.000000e+00


### Kruskal-Wallis Eta-Squared Effect Size

This is used to determine how much impact the shift between windows is present

In [11]:
def calculate_kruskal_eta_sq(data, group_col, value_col):
    # 1. Perform the Kruskal-Wallis H-test
    groups = [group[value_col].values for name, group in data.groupby(group_col)]
    h_stat, p_val = stats.kruskal(*groups)

    n = len(data)
    k = len(groups)

    # 3. Calculate Eta-squared
    eta_sq = (h_stat - k + 1) / (n - k)

    return h_stat, p_val, eta_sq

h, p, effect_size = calculate_kruskal_eta_sq(clean_df, 'window', 'sentiment_score')

print(f"Eta-squared: {effect_size:.4f}")

Eta-squared: 0.0495


Median is computed to see which of the windows differs the most

In [12]:
median_sentiment_by_window = clean_df.groupby('window')['sentiment_score'].median()
display(median_sentiment_by_window)

,sentiment_score
window,
cancel,0.834741
post,0.965690
pre,0.846708
